# SFSC 2026 — V5 Step 4: 4-fold seed -> crop refiner, test refinement, pooled scoring (Colab)

The fold-0 pilot (`SFSC2026_V5_Refiner_Colab.ipynb`) showed better masks (filaments with a matchable candidate 0.710 -> 0.768,
29% of near-misses converted) but almost no PQ gain, because the keep decision was learned from ~130 images. This notebook
trains the refiner on every fold, so every training image gets out-of-fold refined candidates, and scores them the way V4 was
scored: a gradient-boosted scorer cross-fitted over the 4 folds, compared with V4 (0.4550) by paired bootstrap and fold wins.

| Stage (`SFSC_STAGE`) | Runtime | What it does | Output |
|---|---|---|---|
| `train` | GPU | for each fold in `SFSC_FOLDS`: train the refiner on the other folds, refine that fold's images | `fold{f}/best.pt`, `refined_oof_fold{f}.pkl` |
| `test` | GPU | refine every test image with all 4 refiners (probabilities averaged, D4 TTA) | `refined_test.pkl` |
| `score` | CPU | variants `v4` (replica), `v5` (refined only), `union` (refined + V4 candidates on free pixels); cross-fitted scorer; exports | `variants.csv`, `test_export/` |

Seeds = V3 out-of-fold components ∪ V5 detections ∪ V4 detections (score >= 0.01), deduplicated at IoU 0.5; both detectors are
out-of-fold for every training image. New scorer features: V5 detector evidence, refiner peak/mean probability, IoU between the
refined mask and its seed. Two `train` processes (folds 0,1 and 2,3) can share one GPU.


## Cell 01 — Drive access, inputs (read only), output folder, stage


In [ ]:
from pathlib import Path
import os, shutil, subprocess

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_MODE = os.environ.get("SFSC_DRIVE_MODE", "mount" if IN_COLAB else "local")   # mount | rclone | local
RCLONE_REMOTE = os.environ.get("SFSC_RCLONE_REMOTE", "gdrive")
STAGE = os.environ.get("SFSC_STAGE", "train")                                      # train | test | score
LOCAL_ROOT = Path(os.environ.get("SFSC_LOCAL_ROOT", "/content/sfsc_v5_cv" if IN_COLAB else "/tmp/sfsc_v5_cv"))
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA = LOCAL_ROOT / "dataset"
DEBUG = os.environ.get("SFSC_DEBUG", "0") == "1"
OUT_NAME = "refiner_cv_debug" if DEBUG else "refiner_cv"

def rclone(*args, check=True):
    return subprocess.run(["rclone", *args, "--transfers", "16", "--checkers", "16"], check=check)

if DRIVE_MODE == "mount":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/SFSC2026")
elif DRIVE_MODE == "rclone":
    if shutil.which("rclone") is None:
        subprocess.run("curl -fsSL https://rclone.org/install.sh | bash", shell=True, check=True)
    DRIVE_ROOT = LOCAL_ROOT / "drive_mirror"
    remote = f"{RCLONE_REMOTE}:SFSC2026"
    rclone("copy", f"{remote}/data", str(DRIVE_ROOT / "data"), "--include", "/sfsc2026-data.zip")
    rclone("copy", f"{remote}/outputs/sfsc_unet_v3", str(DRIVE_ROOT / "outputs" / "sfsc_unet_v3"),
           "--include", "/*.{csv,json}", "--include", "/fold*/{config,done}.json", "--include", "/oof/**",
           "--include", "/test_probs/**")
    rclone("copy", f"{remote}/outputs/sfsc_fusion_v4", str(DRIVE_ROOT / "outputs" / "sfsc_fusion_v4"),
           "--include", "/*.json", "--include", "/detections/**", "--include", "/detector/fold*/done.json")
    rclone("copy", f"{remote}/outputs/sfsc_v5/seeds", str(DRIVE_ROOT / "outputs" / "sfsc_v5" / "seeds"),
           "--include", "/detections/**", "--include", "/detector/fold*/done.json")
    rclone("copy", f"{remote}/outputs/sfsc_v5/step0", str(DRIVE_ROOT / "outputs" / "sfsc_v5" / "step0"),
           "--include", "/manifest_step0.json", check=False)
    rclone("copy", f"{remote}/outputs/sfsc_v5/{OUT_NAME}", str(DRIVE_ROOT / "outputs" / "sfsc_v5" / OUT_NAME),
           "--include", "/fold*/{best.pt,done.json}", "--include", "/refined_*.pkl", check=False)   # resume state
else:
    DRIVE_ROOT = Path(os.environ["SFSC_DRIVE_ROOT"])

DATA_ZIP = DRIVE_ROOT / "data" / "sfsc2026-data.zip"
DRIVE_EXTRACTED = DRIVE_ROOT / "data" / "SFSCDataset"
V3_RUN = DRIVE_ROOT / "outputs" / "sfsc_unet_v3"
V4_RUN = DRIVE_ROOT / "outputs" / "sfsc_fusion_v4"
STEP0_RUN = DRIVE_ROOT / "outputs" / "sfsc_v5" / "step0"
SEEDS_RUN = DRIVE_ROOT / "outputs" / "sfsc_v5" / "seeds"
RUN = DRIVE_ROOT / "outputs" / "sfsc_v5" / OUT_NAME
RUN.mkdir(parents=True, exist_ok=True)

def push(sub=""):
    if DRIVE_MODE == "rclone":
        rclone("copy", str(RUN / sub), f"{RCLONE_REMOTE}:SFSC2026/outputs/sfsc_v5/{OUT_NAME}/{sub}".rstrip("/"), check=False)

print("Drive mode:", DRIVE_MODE, "| stage:", STAGE, "| output:", RUN, "| debug:", DEBUG)


## Cell 02 — Dependencies


In [ ]:
import importlib.util, sys
REQUIRED = {"segmentation_models_pytorch": "segmentation-models-pytorch>=0.3.4", "timm": "timm>=0.9",
            "sklearn": "scikit-learn>=1.3", "pycocotools": "pycocotools>=2.0.7", "cv2": "opencv-python-headless>=4.8",
            "scipy": "scipy>=1.10", "pandas": "pandas>=2.0", "tqdm": "tqdm>=4.66"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
import torch, segmentation_models_pytorch as smp
print("torch", torch.__version__, "| smp", smp.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## Cell 03 — Imports, configuration (edit here), settings inherited from V3 / V4 / Step 0, helpers


In [ ]:
import copy, gc, hashlib, json, math, multiprocessing as mp, pickle, platform
import random, re, shutil, subprocess, sys, tempfile, time, warnings, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from dataclasses import dataclass, asdict
from functools import lru_cache

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as mask_api
from sklearn.ensemble import HistGradientBoostingRegressor
from tqdm.auto import tqdm


@dataclass
class Config:
    debug_run: bool = DEBUG
    local_dataset_override: str | None = os.environ.get("SFSC_DATASET") or None
    skip_empty_records: bool = True
    stage: str = STAGE
    folds: tuple = tuple(int(f) for f in os.environ.get("SFSC_FOLDS", "0,1,2,3").split(","))

    # ---- seeds ----
    seed_det_floor: float = 0.01             # detections at or above this score become seeds
    seed_dedup_iou: float = 0.5
    match_min_iou: float = 0.1               # a ground-truth instance belongs to a seed's target from this IoU

    # ---- crops ----
    crop_size: int = 320                     # network input (px)
    context: float = 1.8                     # crop side = context x longer box side
    min_crop: int = 128                      # native px
    max_crop: int = 1024
    jitter_seed_px: int = 4                  # ground-truth seeds: shift up to this many px
    jitter_morph_px: int = 2                 # ...and dilate/erode up to this many px
    spine_width: int = 3

    # ---- training ----
    encoder: str = "tu-tf_efficientnetv2_s.in21k_ft_in1k"
    epochs: int = 24
    samples_per_epoch: int = 16000
    gt_seed_fraction: float = 0.5
    batch: int = 64
    lr: float = 4e-4
    weight_decay: float = 1e-4
    spine_weight: float = 0.3
    workers: int = int(os.environ.get("SFSC_WORKERS", "22"))

    # ---- inference / decision ----
    prob_threshold: float = 0.5              # relative: mask = prob >= this x the crop's peak probability
    min_peak: float = 0.1                    # seeds whose refined peak is below this are dropped
    dup_keep_frac: float = 0.5               # a refined mask is a duplicate if less than this fraction is still free
    min_area: int = 32
    tta: tuple = ("r0", "r1", "r2", "r3")
    union_keep_frac: float = 0.5             # a V4 candidate joins the union if this fraction of it is still free
    min_gain: float = 0.005                  # adoption rule against the V4 replica...
    min_fold_wins: int = 3                   # ...with wins in at least this many folds
    bootstrap: int = 2000
    never_empty: bool = True

CFG = Config()
if CFG.debug_run:
    CFG.epochs, CFG.samples_per_epoch = 2, 1024
DEBUG_LIMIT = None                               # V4's staging code reads it; the pilot subsets images itself

_cfg_files = sorted(V3_RUN.glob("fold*/config.json"))
V3CFG = json.loads(_cfg_files[0].read_text())["config"]
for _k in ("seed", "n_folds", "disk_radius_fraction", "radial_bins", "flat_median_kernel", "flat_std", "contact_radius",
           "connectivity", "fill_holes", "min_core_area", "contact_assign_radius", "scorer_prefilter_max",
           "scorer_prefilter_area"):
    setattr(CFG, _k, V3CFG[_k])
CFG.expected_shape = tuple(int(x) for x in V3CFG["expected_shape"])
CFG.scorer_taus = tuple(float(t) for t in V3CFG["scorer_taus"])
V4CFG = json.loads((V4_RUN / "manifest_v4.json").read_text())["config"]
for _k in ("det_min_area", "det_link_frac", "det_only_floor", "det_only_max_claimed", "det_grow", "det_claim_t",
           "det_claim_min_frac", "det_only_min_area"):
    setattr(CFG, _k, V4CFG[_k])
STEP0 = json.loads((STEP0_RUN / "manifest_step0.json").read_text()) if (STEP0_RUN / "manifest_step0.json").is_file() else None

random.seed(CFG.seed); np.random.seed(CFG.seed)
cv2.setNumThreads(0)
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 2)
CPU_POOL = int(os.environ.get("SFSC_CPU_POOL", max(2, CPU_COUNT - 2)))
CFG.workers = min(CFG.workers, max(2, CPU_COUNT - 2))
T0 = time.time()

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1 << 20), b""):
            h.update(part)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()

def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".json_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(obj, f, indent=2, default=str)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def save_npy_atomic(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".npy_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "wb") as f:
            np.save(f, value, allow_pickle=False)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def copy_atomic(src, dst):
    """Copy unless a file of the same size is already there."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_file() and dst.stat().st_size == src.stat().st_size:
        return
    copy_force(src, dst)

def copy_force(src, dst):
    """Always copy (checkpoints of the same size differ in content)."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = dst.with_suffix(dst.suffix + ".part")
    shutil.copyfile(src, temp)
    os.replace(temp, dst)

def save_pickle(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".part")
    with open(temp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temp, path)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

print("CPU cores:", CPU_COUNT, "| Step 0 final:", None if STEP0 is None else (STEP0["final"], STEP0["final_variant"]))


## Cell 04 — Stage the dataset (unchanged from V4)


In [ ]:
ANNOTATION_BASENAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
LOCAL_ZIP = LOCAL_ROOT / "sfsc_dataset.zip"

def locate_dataset(root):
    root = Path(root)
    hits = sorted(root.rglob(ANNOTATION_BASENAME)) if root.is_dir() else []
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one {ANNOTATION_BASENAME} under {root}; found {len(hits)}")
    ann = hits[0]
    train_dir = ann.parent / "train_images"
    test_dir = ann.parent.parent / "test" / "test_images"
    if not (train_dir.is_dir() and test_dir.is_dir()):
        raise FileNotFoundError(f"Missing train_images or test_images near {ann}")
    return ann, train_dir, test_dir

def image_paths(folder):
    return {p.name: p for p in Path(folder).iterdir()
            if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg")}

def staged_and_ready(root):
    try:
        a, t, s = locate_dataset(root)
        return a.stat().st_size > 1000 and len(image_paths(t)) > 0 and len(image_paths(s)) > 0
    except (FileNotFoundError, OSError):
        return False

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    resolved_root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in tqdm(archive.infolist(), desc="Extract dataset"):
            if member.is_dir():
                continue
            dest = (destination / member.filename).resolve()
            if not dest.is_relative_to(resolved_root):
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
            if dest.is_file() and dest.stat().st_size == member.file_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            archive.extract(member, destination)

if CFG.local_dataset_override:
    DATA_ROOT = Path(CFG.local_dataset_override)
    if not staged_and_ready(DATA_ROOT):
        raise FileNotFoundError(f"Dataset override not ready: {DATA_ROOT}")
elif staged_and_ready(LOCAL_DATA):
    DATA_ROOT = LOCAL_DATA
else:
    DATA_ROOT = LOCAL_DATA
    if DATA_ZIP.is_file():
        print("Copying ZIP to local SSD once per runtime...")
        copy_atomic(DATA_ZIP, LOCAL_ZIP)
        safe_extract(LOCAL_ZIP, DATA_ROOT)
    elif DRIVE_EXTRACTED.is_dir():
        print("Copying extracted dataset folder from Drive to local SSD...")
        shutil.copytree(DRIVE_EXTRACTED, DATA_ROOT, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(f"Expected {DATA_ZIP} or {DRIVE_EXTRACTED}. Or set CFG.local_dataset_override.")
if not staged_and_ready(DATA_ROOT):
    raise RuntimeError("Local staging incomplete. Check ZIP contents and free disk space.")

ANN_PATH, TRAIN_DIR, TEST_DIR = locate_dataset(DATA_ROOT)
TRAIN_PATHS, TEST_PATHS = image_paths(TRAIN_DIR), image_paths(TEST_DIR)
for label, paths in (("train", TRAIN_PATHS), ("test", TEST_PATHS)):
    if len({Path(n).stem for n in paths}) != len(paths):
        raise ValueError(f"Duplicate {label} image stems; IDs would collide.")

def read_gray(path):
    with Image.open(path) as im:
        if im.mode == "L":
            return np.asarray(im, dtype=np.uint8).copy()
        if im.mode != "RGB":
            raise ValueError(f"Unexpected image mode {im.mode}: {path}")
        arr = np.asarray(im, dtype=np.uint8)
        d1 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 1])
        d2 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 2])
        if d1.max() > 2 or d2.max() > 2:
            raise ValueError(f"Non-grayscale RGB image: {path}")
        return arr[:, :, 0].copy()

RAW = json.loads(ANN_PATH.read_text())
ANN_SHA = hash_file(ANN_PATH)
print("Annotation JSON:", ANN_PATH, "| sha256", ANN_SHA[:16])
print("JPEGs:", len(TRAIN_PATHS), "train /", len(TEST_PATHS), "test | free local GiB:",
      round(shutil.disk_usage(LOCAL_ROOT).free / 2**30, 1))


## Cell 05 — COCO parsing, record index (unchanged from V4)


In [ ]:
H, W = CFG.expected_shape

def native_rle(rle):
    counts = rle["counts"]
    if isinstance(counts, str):
        counts = counts.encode("ascii")
    return {"size": [int(x) for x in rle["size"]], "counts": counts}

def json_rle(rle):
    counts = rle["counts"]
    return {"size": [int(x) for x in rle["size"]],
            "counts": counts.decode("ascii") if isinstance(counts, bytes) else counts}

def decode_mask(rle):
    a = mask_api.decode(native_rle(rle))
    return np.any(a, axis=2) if a.ndim == 3 else (a > 0)

def segmentation_to_rle(seg, h, w):
    if isinstance(seg, dict):
        if list(seg.get("size", [h, w])) != [h, w]:
            raise ValueError("Annotation RLE shape mismatch")
        rle = (mask_api.frPyObjects(seg, h, w) if isinstance(seg.get("counts"), list)
               else native_rle({"size": [h, w], "counts": seg["counts"]}))
    elif isinstance(seg, list) and len(seg):
        polys = seg if isinstance(seg[0], (list, tuple)) else [seg]
        if any(len(p) < 6 or len(p) % 2 for p in polys):
            raise ValueError("Malformed polygon segmentation")
        rle = mask_api.merge(mask_api.frPyObjects(polys, h, w))
    else:
        raise ValueError("Unsupported empty/malformed segmentation")
    return json_rle(rle)

STEMS = defaultdict(list)
for _n in TRAIN_PATHS:
    STEMS[Path(_n).stem].append(_n)

def canonical_filename(s):
    name = Path(str(s)).name
    if name in TRAIN_PATHS:
        return name
    cands = STEMS.get(Path(name).stem, [])
    if len(cands) == 1:
        return cands[0]
    raise ValueError(f"Unresolvable annotated JPEG: {s}")

IMAGES_BY_ID = {}
for im in RAW["images"]:
    rid = str(im["id"])
    if rid in IMAGES_BY_ID:
        raise ValueError(f"Duplicate image record {rid}")
    if (int(im["height"]), int(im["width"])) != (H, W):
        raise ValueError(f"Unexpected dimensions for record {rid}")
    IMAGES_BY_ID[rid] = im

ANNOTATIONS_BY_ID = defaultdict(list)
_seen = set()
for a in RAW["annotations"]:
    aid, rid = str(a["id"]), str(a["image_id"])
    if aid in _seen or rid not in IMAGES_BY_ID:
        raise ValueError(f"Duplicate annotation or missing image: {aid} / {rid}")
    if int(a.get("iscrowd", 0)):
        raise ValueError(f"Unexpected crowd annotation {aid}")
    if int(a.get("category_id", 1)) not in (1, 2, 3, 4):
        raise ValueError(f"Unexpected category {a.get('category_id')} in {aid}")
    _seen.add(aid)
    ANNOTATIONS_BY_ID[rid].append(a)

RECORD_IDS_BY_FILE = defaultdict(list)
for rid, im in IMAGES_BY_ID.items():
    RECORD_IDS_BY_FILE[canonical_filename(im["file_name"])].append(rid)
for k in RECORD_IDS_BY_FILE:
    RECORD_IDS_BY_FILE[k].sort()
UNIQUE_FILES = sorted(RECORD_IDS_BY_FILE)

FILE_HASHES = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    for name, digest in zip(UNIQUE_FILES, pool.map(lambda n: hash_file(TRAIN_PATHS[n]), UNIQUE_FILES)):
        FILE_HASHES[name] = digest

unannotated = sorted(set(TRAIN_PATHS) - set(UNIQUE_FILES))
if unannotated:
    warnings.warn(f"{len(unannotated)} train JPEGs have no annotation record; they are not used.")
print("Annotated JPEGs:", len(UNIQUE_FILES), "| annotator records:", len(IMAGES_BY_ID),
      "| instances:", len(RAW["annotations"]))
print("Records per JPEG:", pd.Series([len(v) for v in RECORD_IDS_BY_FILE.values()]).value_counts().sort_index().to_dict())


## Cell 06 — Disk geometry, flattening, image caches, ground-truth index (unchanged from V4) + spines


In [ ]:
CY, CX = (H - 1) / 2.0, (W - 1) / 2.0
DISK_R = CFG.disk_radius_fraction * min(H, W) / 2.0
_YY, _XX = np.mgrid[0:H, 0:W]
RADIAL = (np.hypot(_YY - CY, _XX - CX) / DISK_R).astype(np.float32)
YY_FLAT = _YY.ravel().astype(np.float64)
XX_FLAT = _XX.ravel().astype(np.float64)
del _YY, _XX
DISK = RADIAL < 1.0
RBIN = np.minimum((RADIAL * CFG.radial_bins).astype(np.int32), CFG.radial_bins - 1)
RBIN_DISK = RBIN[DISK]

IMG_MEAN, IMG_STD = 0.5, 0.25
RAD_MEAN, RAD_STD, RAD_CLIP = 0.6, 0.35, 1.25

def flatten_disk(img):
    """Divide by the radial limb-darkening profile, then by a large local median; off-disk = neutral (1.0 -> 128)."""
    x = img.astype(np.float32)
    nb = CFG.radial_bins
    s = np.bincount(RBIN_DISK, weights=x[DISK], minlength=nb)
    c = np.bincount(RBIN_DISK, minlength=nb)
    valid = c > 0
    prof = np.interp(np.arange(nb), np.flatnonzero(valid), (s[valid] / c[valid]))
    k = 7
    prof = np.convolve(np.pad(prof, k // 2, mode="edge"), np.ones(k) / k, mode="valid")
    ratio = x / np.maximum(prof[RBIN], 1.0).astype(np.float32)
    ratio[~DISK] = 1.0
    small = cv2.resize(ratio, (W // 4, H // 4), interpolation=cv2.INTER_AREA)
    small_u8 = np.clip(small * 128.0, 0, 255).astype(np.uint8)
    ksize = max(3, int(round(CFG.flat_median_kernel * H / 2048)) | 1)
    med = cv2.medianBlur(small_u8, ksize).astype(np.float32) / 128.0
    med = cv2.resize(med, (W, H), interpolation=cv2.INTER_LINEAR)
    flat = ratio / np.maximum(med, 0.25)
    flat[~DISK] = 1.0
    return np.clip(flat * 128.0, 0, 255).astype(np.uint8)

CACHE = LOCAL_ROOT / "cache" / f"v3_{ANN_SHA[:12]}_{H}x{W}_r{CFG.disk_radius_fraction:.4f}_k{CFG.flat_median_kernel}_c{CFG.contact_radius}"
for sub in ("img", "flat", "soft", "contact", "gt"):
    (CACHE / sub).mkdir(parents=True, exist_ok=True)

def cache_key(split, name):
    return f"{split}__{Path(name).stem}"

def cache_image(split, name, path):
    key = cache_key(split, name)
    ip, fp = CACHE / "img" / f"{key}.npy", CACHE / "flat" / f"{key}.npy"
    if ip.is_file() and fp.is_file():
        return
    img = read_gray(path)
    if img.shape != (H, W):
        raise ValueError(f"Wrong JPEG shape {img.shape}: {path}")
    save_npy_atomic(ip, img)
    save_npy_atomic(fp, flatten_disk(img))
def build_gt(name):
    """Flat pixel indices and areas of every annotated filament, per annotator set."""
    gp = CACHE / "gt" / f"{cache_key('train', name)}_v4.pkl"
    if gp.is_file():
        with open(gp, "rb") as f:
            return pickle.load(f)
    records = []
    for rid in RECORD_IDS_BY_FILE[name]:
        idx_list, areas = [], []
        for ann in ANNOTATIONS_BY_ID[rid]:
            m = decode_mask(segmentation_to_rle(ann["segmentation"], H, W))
            if not m.any():
                raise ValueError(f"Empty filament annotation {ann['id']}")
            flat_idx = np.flatnonzero(m).astype(np.int32)
            idx_list.append(flat_idx); areas.append(len(flat_idx))
        records.append({"id": rid, "idx": idx_list, "area": np.asarray(areas, np.float64)})
    gt = {"name": name, "records": records}
    save_pickle(gp, gt)
    return gt

with ThreadPoolExecutor(max_workers=min(8, CPU_COUNT)) as pool:
    list(tqdm(pool.map(lambda n: cache_image("train", n, TRAIN_PATHS[n]), UNIQUE_FILES),
              total=len(UNIQUE_FILES), desc="Cache train images + flattened"))
    list(tqdm(pool.map(lambda n: cache_image("test", n, TEST_PATHS[n]), sorted(TEST_PATHS)),
              total=len(TEST_PATHS), desc="Cache test images + flattened"))
    _gt_list = list(tqdm(pool.map(build_gt, UNIQUE_FILES), total=len(UNIQUE_FILES), desc="Ground-truth index"))

GT_INDEX, N_EMPTY_RECORDS = {}, 0
for g in _gt_list:
    scored = [r for r in g["records"] if len(r["idx"])] if CFG.skip_empty_records else list(g["records"])
    N_EMPTY_RECORDS += len(g["records"]) - len(scored)
    GT_INDEX[g["name"]] = {"name": g["name"], "records": scored, "n_all": len(g["records"])}
del _gt_list

@lru_cache(maxsize=64)
def cached_arrays(key, with_targets=True):
    """(image, flattened image) as memory maps. The second argument is kept for compatibility with V3's functions."""
    return tuple(np.load(CACHE / s / f"{key}.npy", mmap_mode="r") for s in ("img", "flat"))

print("Scored annotator sets:", sum(len(g["records"]) for g in GT_INDEX.values()),
      "| filaments:", sum(len(r["idx"]) for g in GT_INDEX.values() for r in g["records"]),
      "| annotator sets without filaments (skipped: %s): %d" % (CFG.skip_empty_records, N_EMPTY_RECORDS))

def parse_spine(s):
    if isinstance(s, str):
        s = json.loads(s)
    a = np.asarray(s, np.float64).ravel()
    return a.reshape(-1, 2) if a.size >= 4 and a.size % 2 == 0 else np.zeros((0, 2))

# spines in the same order as GT_INDEX[name]["records"][r]["idx"][j]
SPINES = {}
for name in UNIQUE_FILES:
    recs = []
    for rid in RECORD_IDS_BY_FILE[name]:
        anns = ANNOTATIONS_BY_ID[rid]
        if CFG.skip_empty_records and not anns:
            continue
        recs.append([parse_spine(a.get("spine", [])) for a in anns])
    SPINES[name] = recs
assert all(len(SPINES[n]) == len(GT_INDEX[n]["records"]) for n in UNIQUE_FILES)


## Cell 07 — V3 folds, recipe, out-of-fold maps and test maps (unchanged from V4)


In [ ]:
FOLDS = pd.read_csv(V3_RUN / "folds_v3.csv", dtype={"file_name": str, "group": str, "v1_split": str})
if set(FOLDS.file_name) != set(UNIQUE_FILES):
    raise ValueError("V3's folds_v3.csv does not match this dataset.")
FOLDS["fold"] = FOLDS["fold"].astype(int)
FOLD_OF = dict(zip(FOLDS.file_name, FOLDS.fold))

V3_FROZEN = json.loads((V3_RUN / "frozen_recipe_v3.json").read_text())
V3_REC, V3_SHA8 = V3_FROZEN["recipe"], V3_FROZEN["sha256"][:8]
_g = V3_REC["geometry"]
GEOM = (float(_g["t_low"]), None if _g["contact"] is None else float(_g["contact"]), int(_g["offset"]))
V3_TAU = None if V3_REC.get("scorer_tau") is None else float(V3_REC["scorer_tau"])
V3_FOLDS = sorted(int(f) for f in V3_REC["folds"])
OOF_TAG = object_hash({"tta": list(V3CFG["tta_final"]), "tile": V3CFG["tile"], "stride": V3CFG["stride"]})[:10]
MAPS_LOCAL = LOCAL_ROOT / "v3_maps"

def stage_npz(task):
    """Drive .npz (fg, ct) -> two local .npy files that CPU workers can memory-map."""
    src, lf, lc = task
    if not (lf.is_file() and lc.is_file()):
        with np.load(src) as z:
            fg, ct = z["fg"], z["ct"]
        save_npy_atomic(lf, fg); save_npy_atomic(lc, ct)
    return str(lf), str(lc)

def find_oof_dir(fold):
    sha = json.loads((V3_RUN / f"fold{fold}" / "done.json").read_text())["best_sha256"][:10]
    exact = V3_RUN / "oof" / f"fold{fold}_{sha}_{OOF_TAG}"
    if exact.is_dir():
        return exact
    cands = sorted((d for d in (V3_RUN / "oof").glob(f"fold{fold}_*") if d.is_dir()),
                   key=lambda d: len(list(d.glob("*.npz"))))
    if not cands:
        raise FileNotFoundError(f"No out-of-fold maps for fold {fold} under {V3_RUN / 'oof'}. Run V3's Cell 19.")
    return cands[-1]

_tasks, _owners = [], []
for fold in V3_FOLDS:
    d = find_oof_dir(fold)
    names = sorted(FOLDS.loc[FOLDS.fold == fold, "file_name"])
    if DEBUG_LIMIT:
        names = names[:DEBUG_LIMIT]
    for name in names:
        stem = Path(name).stem
        src = d / f"{stem}.npz"
        if not src.is_file():
            raise FileNotFoundError(f"Missing V3 out-of-fold map {src}. Run V3's Cell 19 to completion.")
        _tasks.append((src, MAPS_LOCAL / d.name / f"{stem}_fg.npy", MAPS_LOCAL / d.name / f"{stem}_ct.npy"))
        _owners.append(name)
with ThreadPoolExecutor(max_workers=8) as pool:
    OOF_PATHS = dict(zip(_owners, tqdm(pool.map(stage_npz, _tasks), total=len(_tasks), desc="Stage V3 out-of-fold maps")))

_tasks, _owners = [], []
for name in sorted(TEST_PATHS)[:DEBUG_LIMIT] if DEBUG_LIMIT else sorted(TEST_PATHS):
    stem = Path(name).stem
    src = V3_RUN / "test_probs" / f"{stem}_{V3_SHA8}.npz"
    if not src.is_file():
        raise FileNotFoundError(f"Missing V3 test map {src}. Run V3's Cell 23 to completion.")
    _tasks.append((src, MAPS_LOCAL / f"test_{V3_SHA8}" / f"{stem}_fg.npy", MAPS_LOCAL / f"test_{V3_SHA8}" / f"{stem}_ct.npy"))
    _owners.append(name)
with ThreadPoolExecutor(max_workers=8) as pool:
    TEST_MAPS = dict(zip(_owners, tqdm(pool.map(stage_npz, _tasks), total=len(_tasks), desc="Stage V3 test maps")))

print("V3 geometry:", GEOM, "| V3 keep threshold:", V3_TAU, "| V3 cross-fitted out-of-fold PQ:", V3_REC.get("oof_pq_scorer"))
print("V3 maps staged:", len(OOF_PATHS), "out-of-fold,", len(TEST_MAPS), "test")


## Cell 08 — Exact PQ engine and instance extraction (unchanged from V4)


In [ ]:
STRUCT8 = ndi.generate_binary_structure(2, 2)
STRUCT4 = ndi.generate_binary_structure(2, 1)
CONN = STRUCT8 if CFG.connectivity == 8 else STRUCT4
FEATURES = ("log_area", "mean_p", "max_p", "frac50", "frac70", "frac90", "mean_contact",
            "elongation", "major_len", "radial", "mean_img", "mean_flat")
F_IDX = {k: i for i, k in enumerate(FEATURES)}

def qt(t):
    return int(round(float(t) * 255))

def match_stats(iou):
    """Strict IoU>0.5, one-to-one max-IoU assignment. Returns (sum_iou, tp, fp, fn, records)."""
    n_gt, n_pred = iou.shape
    if n_gt == 0 or n_pred == 0:
        return np.array([0.0, 0, n_pred, n_gt, 1], np.float64)
    q = iou > 0.5
    if not q.any():
        return np.array([0.0, 0, n_pred, n_gt, 1], np.float64)
    rows, cols = np.flatnonzero(q.any(1)), np.flatnonzero(q.any(0))
    sub = np.where(q[np.ix_(rows, cols)], iou[np.ix_(rows, cols)], 0.0)
    r, c = linear_sum_assignment(sub, maximize=True)
    v = sub[r, c]
    v = v[v > 0]
    tp = len(v)
    return np.array([float(v.sum()), tp, n_pred - tp, n_gt - tp, 1], np.float64)

def match_pairs(iou):
    if iou.size == 0 or not (iou > 0.5).any():
        return []
    sub = np.where(iou > 0.5, iou, 0.0)
    r, c = linear_sum_assignment(sub, maximize=True)
    return [(int(a), int(b), float(sub[a, b])) for a, b in zip(r, c) if sub[a, b] > 0]

def summarize(st):
    s, tp, fp, fn, rec = [float(x) for x in st]
    den = tp + 0.5 * (fp + fn)
    return {"pq": s / den if den else 1.0, "sq": s / tp if tp else 0.0, "rq": tp / den if den else 1.0,
            "precision": tp / (tp + fp) if tp + fp else 0.0, "recall": tp / (tp + fn) if tp + fn else 0.0,
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "records": int(rec)}

def relabel(lab):
    if lab.max() == 0:
        return np.zeros(lab.shape, np.int32), 0
    areas = np.bincount(lab.ravel())
    ids = np.flatnonzero(areas[1:]) + 1
    lut = np.zeros(len(areas), np.int32)
    lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
    return lut[lab], len(ids)

def fill_label_holes(lab):
    fg = lab > 0
    holes = ndi.binary_fill_holes(fg) & ~fg
    if not holes.any():
        return lab
    hl, _ = ndi.label(holes, structure=STRUCT4)
    out = lab.copy()
    for i, sl in enumerate(ndi.find_objects(hl), 1):
        if sl is None:
            continue
        esl = (slice(max(sl[0].start - 1, 0), sl[0].stop + 1), slice(max(sl[1].start - 1, 0), sl[1].stop + 1))
        hole = hl[esl] == i
        ring = ndi.binary_dilation(hole, STRUCT8) & ~hole
        vals = lab[esl][ring]
        vals = vals[vals > 0]
        if vals.size:
            out[esl][hole] = np.bincount(vals).argmax()
    return out

def geometry_labels(fg, ct, geom):
    t_low, contact_t, offset = geom
    low = fg >= qt(t_low)
    if contact_t is not None:
        cores = low & (ct < qt(contact_t))
        lab, n = ndi.label(cores, structure=CONN)
        if n:
            areas = np.bincount(lab.ravel(), minlength=n + 1)
            small = areas < CFG.min_core_area
            small[0] = False
            if small.any():
                lab[small[lab]] = 0
        if lab.any():
            dist, (iy, ix) = ndi.distance_transform_edt(lab == 0, return_distances=True, return_indices=True)
            lab = np.where(low & (dist <= CFG.contact_assign_radius), lab[iy, ix], 0).astype(np.int32)
        else:
            lab = np.zeros((fg.shape), np.int32)
    else:
        lab, _ = ndi.label(low, structure=CONN)
        lab = lab.astype(np.int32)
    if CFG.fill_holes and lab.any():
        lab = fill_label_holes(lab)
    for _ in range(max(offset, 0)):
        lab = np.where(lab == 0, ndi.grey_dilation(lab, footprint=STRUCT4), lab)
    for _ in range(max(-offset, 0)):
        lab = np.where(ndi.grey_erosion(lab, footprint=STRUCT4) == lab, lab, 0)
    return relabel(lab)

def component_features(lab, n, fg, ct, img, flat):
    if n == 0:
        return np.zeros((0, len(FEATURES)), np.float32), np.zeros(0, np.float64)
    L = lab.ravel()
    nz = np.flatnonzero(L)
    Lz = L[nz]
    def s(w):
        return np.bincount(Lz, weights=w, minlength=n + 1)[1:]
    area = np.bincount(Lz, minlength=n + 1)[1:].astype(np.float64)
    a = np.maximum(area, 1.0)
    fgz = np.asarray(fg).ravel()[nz]
    p = fgz.astype(np.float64) / 255.0
    mx = np.zeros(n + 1, np.float64)
    np.maximum.at(mx, Lz, fgz.astype(np.float64))
    yz, xz = YY_FLAT[nz], XX_FLAT[nz]
    my, mxx = s(yz) / a, s(xz) / a
    syy = np.maximum(s(yz * yz) / a - my ** 2, 0)
    sxx = np.maximum(s(xz * xz) / a - mxx ** 2, 0)
    sxy = s(yz * xz) / a - my * mxx
    tr, det = syy + sxx, syy * sxx - sxy ** 2
    disc = np.sqrt(np.maximum(tr ** 2 / 4 - det, 0))
    l1, l2 = tr / 2 + disc, np.maximum(tr / 2 - disc, 1e-3)
    feats = np.stack([
        np.log1p(area), s(p) / a, mx[1:] / 255.0,
        s((fgz >= qt(0.5)).astype(np.float64)) / a, s((fgz >= qt(0.7)).astype(np.float64)) / a,
        s((fgz >= qt(0.9)).astype(np.float64)) / a,
        s(np.asarray(ct).ravel()[nz].astype(np.float64) / 255.0) / a,
        np.sqrt((l1 + 1e-3) / l2), 4.0 * np.sqrt(l1),
        np.hypot(my - CY, mxx - CX) / DISK_R,
        s(np.asarray(img).ravel()[nz].astype(np.float64) / 255.0) / a,
        s(np.asarray(flat).ravel()[nz].astype(np.float64) / 128.0) / a,
    ], 1).astype(np.float32)
    return feats, area

def overlap_matrices(lab, n, area, gt):
    L = lab.ravel()
    out = []
    for rec in gt["records"]:
        inter = np.zeros((len(rec["idx"]), n), np.float64)
        if n:
            for j, idx in enumerate(rec["idx"]):
                inter[j] = np.bincount(L[idx], minlength=n + 1)[1:]
        union = rec["area"][:, None] + area[None, :] - inter
        iou = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        out.append((inter, iou))
    return out

def grid_keep(feats, area, flt):
    th, mc, ma = flt
    if len(area) == 0:
        return np.zeros(0, bool)
    return ((np.rint(feats[:, F_IDX["max_p"]] * 255) >= qt(th)) & (feats[:, F_IDX["mean_p"]] >= mc - 1e-9)
            & (area >= ma))

def prefilter_keep(feats, area):
    if len(area) == 0:
        return np.zeros(0, bool)
    return ((np.rint(feats[:, F_IDX["max_p"]] * 255) >= qt(CFG.scorer_prefilter_max))
            & (area >= CFG.scorer_prefilter_area))

def load_maps(fg, ct):
    if isinstance(fg, (str, Path)):
        fg = np.load(fg, mmap_mode="r")
    if isinstance(ct, (str, Path)):
        ct = np.load(ct, mmap_mode="r")
    return np.asarray(fg), np.asarray(ct)

# ---- process-pool workers (fork: they see GT_INDEX, CACHE, CFG of the parent at fork time) ----
def score_image_grid(task):
    name, fg, ct, geoms, filters = task
    fg, ct = load_maps(fg, ct)
    img, flat = cached_arrays(cache_key("train", name), False)
    gt = GT_INDEX[name]
    out = np.zeros((len(geoms), len(filters), 5), np.float64)
    for gi, g in enumerate(geoms):
        lab, n = geometry_labels(fg, ct, g)
        feats, area = component_features(lab, n, fg, ct, img, flat)
        mats = overlap_matrices(lab, n, area, gt)
        for fi, flt in enumerate(filters):
            keep = grid_keep(feats, area, flt)
            for _, iou in mats:
                out[gi, fi] += match_stats(iou[:, keep])
    return name, out

def scorer_rows(task):
    name, fg, ct, geom = task
    fg, ct = load_maps(fg, ct)
    img, flat = cached_arrays(cache_key("train", name), False)
    lab, n = geometry_labels(fg, ct, geom)
    feats, area = component_features(lab, n, fg, ct, img, flat)
    mats = overlap_matrices(lab, n, area, GT_INDEX[name])
    y = np.zeros(n, np.float64)
    for _, iou in mats:
        if iou.shape[0] and n:
            b = iou.max(0)
            y += np.where(b > 0.5, b, 0.0)
    y /= max(len(mats), 1)
    pre = prefilter_keep(feats, area)
    return name, feats[pre], area[pre], y[pre], [iou[:, pre] for _, iou in mats]

_POOL = None
def get_pool(refresh=False):
    """Fork-based CPU pool. Refresh after defining new globals that workers must see."""
    global _POOL
    if refresh and _POOL is not None:
        _POOL.shutdown(wait=True)
        _POOL = None
    if _POOL is None:
        _POOL = ProcessPoolExecutor(max_workers=CPU_POOL, mp_context=mp.get_context("fork"))
    return _POOL

def stats_table(total, geoms, filters):
    rows = []
    for gi, g in enumerate(geoms):
        for fi, f in enumerate(filters):
            rows.append({"t_low": g[0], "contact": g[1], "offset": g[2], "t_high": f[0], "min_mean": f[1],
                         "min_area": f[2], **summarize(total[gi, fi])})
    return pd.DataFrame(rows)

def pick_best(table):
    t = table.copy()
    t["_c"] = t["contact"].isna()
    t["_o"] = (t["offset"] == 0)
    t = t.sort_values(["pq", "_c", "_o", "min_area", "min_mean"], ascending=[False, False, False, True, True])
    return t.iloc[0].drop(["_c", "_o"]).to_dict()

def as_geom(row):
    c = row["contact"]
    return (float(row["t_low"]), None if (c is None or (isinstance(c, float) and np.isnan(c))) else float(c), int(row["offset"]))

def as_filter(row):
    return (float(row["t_high"]), float(row["min_mean"]), int(row["min_area"]))

print("PQ engine ready. Features:", FEATURES)


## Cell 09 — Detections (V4 and V5 seeds; out-of-fold for training images, all fold models for test) and V4's fusion


In [ ]:
def load_dets(run):
    """-> ({train name: detections of the fold model that never saw it}, {test name: [detections per fold model]}, folds)"""
    folds = sorted(int(p.parent.name.replace("fold", "")) for p in (run / "detector").glob("fold*/done.json"))
    oof, test = {}, defaultdict(list)
    for f in folds:
        sha = json.loads((run / "detector" / f"fold{f}" / "done.json").read_text())["weights_sha256"][:8]
        oof.update(load_pickle(run / "detections" / f"oof_fold{f}_{sha}.pkl"))
        tp = run / "detections" / f"test_fold{f}_{sha}.pkl"
        if tp.is_file():
            for n, d in load_pickle(tp).items():
                test[n].append(d)
    return oof, dict(test), folds

DET_V4, DET_V4_TEST, V4_DET_FOLDS = load_dets(V4_RUN)
DET_V5, DET_V5_TEST, V5_DET_FOLDS = load_dets(SEEDS_RUN)
ALL_NAMES = sorted(n for n in OOF_PATHS if n in DET_V4 and n in DET_V5)
TEST_NAMES = sorted(n for n in TEST_MAPS if n in DET_V4_TEST and n in DET_V5_TEST)
if CFG.debug_run:
    ALL_NAMES = sorted(sum(([n for n in ALL_NAMES if FOLD_OF[n] == f][:12] for f in range(CFG.n_folds)), []))
    TEST_NAMES = TEST_NAMES[:6]
elif len(V5_DET_FOLDS) != CFG.n_folds or len(ALL_NAMES) != len(UNIQUE_FILES):
    raise RuntimeError(f"V5 seeds must cover all folds and images: folds {V5_DET_FOLDS}, images {len(ALL_NAMES)}")
IMG_FOLD = np.array([FOLD_OF[n] for n in ALL_NAMES])
print("Images:", len(ALL_NAMES), "| test:", len(TEST_NAMES), "| V4 detector folds:", V4_DET_FOLDS,
      "| V5 detector folds:", V5_DET_FOLDS)

DET_FEATURES = ("det_score", "det_iou", "det_cov")
ALL_FEATURES = tuple(FEATURES) + DET_FEATURES + ("src",)
N_V3F, N_DETF = len(FEATURES), len(DET_FEATURES)
_GROW_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * CFG.det_grow + 1, 2 * CFG.det_grow + 1))

def rasterize_detections(dets):
    """Detections of one detector -> list of (score, y0, x0, boolean mask of its bounding window), best first."""
    out = []
    for s, p in zip(dets["scores"], dets["polys"]):
        if p is None or len(p) < 3:
            continue
        q = np.rint(np.asarray(p, np.float64)).astype(np.int32)
        x0, y0 = max(int(q[:, 0].min()), 0), max(int(q[:, 1].min()), 0)
        x1, y1 = min(int(q[:, 0].max()) + 1, W), min(int(q[:, 1].max()) + 1, H)
        if x1 - x0 < 2 or y1 - y0 < 2:
            continue
        roi = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.fillPoly(roi, [q - np.array([x0, y0], np.int32)], 1)
        if int(roi.sum()) < CFG.det_min_area:
            continue
        out.append((float(s), y0, x0, roi.astype(bool)))
    out.sort(key=lambda t: -t[0])
    return out

def detection_evidence(lab, n, area, rast):
    """Per candidate: score of the detection sharing most pixels with it, IoU with that detection, fraction covered."""
    score, iou, cover = np.zeros(n), np.zeros(n), np.zeros(n)
    if n == 0 or not rast:
        return score, iou, cover
    best = np.zeros(n)
    covered = np.zeros(lab.shape, bool)
    for s, y0, x0, m in rast:
        h, w = m.shape
        covered[y0:y0 + h, x0:x0 + w] |= m
        inter = np.bincount(lab[y0:y0 + h, x0:x0 + w][m], minlength=n + 1)[1:].astype(np.float64)
        a = float(m.sum())
        upd = (inter >= CFG.det_link_frac * np.minimum(area, a)) & (inter > best)
        if upd.any():
            best[upd] = inter[upd]
            score[upd] = s
            iou[upd] = inter[upd] / (area[upd] + a - inter[upd])
    cover = np.bincount(lab[covered], minlength=n + 1)[1:] / np.maximum(area, 1.0)
    return score, iou, cover

def largest_piece(mask):
    lab, n = ndi.label(mask, structure=CONN)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    return lab == int(sizes.argmax())

def consensus_scores(rasts):
    """Every detection gets the mean, over all detectors, of the best score among that detector's detections
    that overlap it (IoU >= 0.3). With one detector this is its own score; with four, a detection found by only
    one of them is worth a quarter. Keeps test images (4 detectors) comparable to out-of-fold images (1 detector)."""
    items = [(d, s, y0, x0, m, float(m.sum())) for d, rast in enumerate(rasts) for s, y0, x0, m in rast]
    out = []
    for d, s, y0, x0, m, a in items:
        total = s
        for d2 in range(len(rasts)):
            if d2 == d:
                continue
            best = 0.0
            for s2, v0, u0, m2 in rasts[d2]:
                if s2 <= best:
                    break                                         # sorted by score: nothing better follows
                ya, yb = max(y0, v0), min(y0 + m.shape[0], v0 + m2.shape[0])
                xa, xb = max(x0, u0), min(x0 + m.shape[1], u0 + m2.shape[1])
                if ya >= yb or xa >= xb:
                    continue
                inter = float((m[ya - y0:yb - y0, xa - x0:xb - x0] & m2[ya - v0:yb - v0, xa - u0:xb - u0]).sum())
                if inter >= 0.3 * (a + float(m2.sum()) - inter):
                    best = s2
            total += best
        out.append((total / max(len(rasts), 1), y0, x0, m))
    out.sort(key=lambda t: -t[0])
    return out

def detector_only_candidates(lab, fg, rasts):
    """Objects the detectors found where the UNet has no candidate. The mask is the UNet's probability inside the
    (slightly grown) detection when there is enough of it, otherwise the detection itself. All disjoint."""
    pooled = consensus_scores(rasts)
    occupied = lab > 0
    new = np.zeros(lab.shape, np.int32)
    kinds, g = [], CFG.det_grow
    for s, y0, x0, m in pooled:
        if s < CFG.det_only_floor:
            break
        h, w = m.shape
        if int((m & occupied[y0:y0 + h, x0:x0 + w]).sum()) > CFG.det_only_max_claimed * int(m.sum()):
            continue                                              # already represented
        ya, xa, yb, xb = max(y0 - g, 0), max(x0 - g, 0), min(y0 + h + g, H), min(x0 + w + g, W)
        big = np.zeros((yb - ya, xb - xa), np.uint8)
        big[y0 - ya:y0 - ya + h, x0 - xa:x0 - xa + w] = m
        free = ~occupied[ya:yb, xa:xb]
        det_free = (big > 0) & free
        claim = (cv2.dilate(big, _GROW_KERNEL) > 0) & free & (fg[ya:yb, xa:xb] >= qt(CFG.det_claim_t))
        if int(claim.sum()) >= CFG.det_claim_min_frac * int(det_free.sum()):
            mask, kind = largest_piece(claim), 1                  # drawn by the UNet
        else:
            mask, kind = largest_piece(det_free), 2               # drawn by the detector
        if int(mask.sum()) < CFG.det_only_min_area:
            continue
        kinds.append(kind)
        new[ya:yb, xa:xb][mask] = len(kinds)
        occupied[ya:yb, xa:xb] |= mask
    return new, np.asarray(kinds, np.float32)

def build_candidates(fg, ct, img, flat, det_list, with_detector_only=True):
    """V3 candidates (same geometry and prefilter as V3) + detector-only candidates, with all features.
    Returns (label map, n candidates, n from V3, feature matrix [n, len(ALL_FEATURES)], areas)."""
    lab0, n0 = geometry_labels(fg, ct, GEOM)
    feats, area = component_features(lab0, n0, fg, ct, img, flat)
    pre = prefilter_keep(feats, area)
    lut = np.zeros(n0 + 1, np.int32)
    ids = np.flatnonzero(pre) + 1
    lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
    lab, n_v3 = lut[lab0], len(ids)
    feats, area = feats[pre], area[pre]
    rasts = [rasterize_detections(d) for d in det_list]
    kinds = np.zeros(0, np.float32)
    if with_detector_only and any(rasts):
        new, kinds = detector_only_candidates(lab, fg, rasts)
        if len(kinds):
            lab = np.where(new > 0, new + n_v3, lab)
            feats, area = component_features(lab, n_v3 + len(kinds), fg, ct, img, flat)
    n = n_v3 + len(kinds)
    ev = np.zeros((n, N_DETF))
    for rast in rasts:
        ev += np.stack(detection_evidence(lab, n, area, rast), 1)
    if rasts:
        ev /= len(rasts)
    src = np.r_[np.zeros(n_v3, np.float32), kinds]
    X = np.concatenate([feats.astype(np.float32), ev.astype(np.float32), src[:, None].astype(np.float32)], 1)
    return lab, n, n_v3, X, area


## Cell 10 — Seed pools and targets (split-aware; only the pools the stage needs are built)


In [ ]:
SEED_GEOM = GEOM
if STEP0 is not None and STEP0["final_variant"].get("t_low") is not None:
    SEED_GEOM = (float(STEP0["final_variant"]["t_low"]), GEOM[1], GEOM[2])
SEED_PRE = True if STEP0 is None else bool(STEP0["final_variant"].get("pre", True))

def v3_seeds(fg, ct, img, flat):
    lab, n = geometry_labels(fg, ct, (SEED_GEOM[0], SEED_GEOM[1], 0))
    feats, area = component_features(lab, n, fg, ct, img, flat)
    keep = prefilter_keep(feats, area) if SEED_PRE else area >= CFG.scorer_prefilter_area
    out = []
    for k, sl in enumerate(ndi.find_objects(lab)):
        if sl is None or not keep[k]:
            continue
        out.append((sl[0].start, sl[1].start, lab[sl] == k + 1, float(feats[k, F_IDX["max_p"]]), 0))
    return out

def window_iou(a, b):
    (ya, xa, ma), (yb, xb, mb) = a, b
    y0, y1 = max(ya, yb), min(ya + ma.shape[0], yb + mb.shape[0])
    x0, x1 = max(xa, xb), min(xa + ma.shape[1], xb + mb.shape[1])
    if y0 >= y1 or x0 >= x1:
        return 0.0
    i = float((ma[y0 - ya:y1 - ya, x0 - xa:x1 - xa] & mb[y0 - yb:y1 - yb, x0 - xb:x1 - xb]).sum())
    return i / (float(ma.sum()) + float(mb.sum()) - i) if i else 0.0

def maps_for(name, split):
    fg, ct = load_maps(*(OOF_PATHS if split == "train" else TEST_MAPS)[name])
    img, flat = cached_arrays(cache_key(split, name), False)
    return fg, ct, img, flat

def det_lists_for(name, split):
    return [DET_V5[name], DET_V4[name]] if split == "train" else DET_V5_TEST[name] + DET_V4_TEST[name]

def seed_pool(name, split, floor):
    fg, ct, img, flat = maps_for(name, split)
    seeds = v3_seeds(fg, ct, img, flat)
    for dets in det_lists_for(name, split):
        for s, y0, x0, m in rasterize_detections(dets):
            if s < floor:
                break
            cand = (y0, x0, m)
            if all(window_iou(cand, (a, b, c)) < CFG.seed_dedup_iou for a, b, c, _, _ in seeds):
                seeds.append((y0, x0, m, float(s), 1))
    return seeds

def gt_window_iou(win, idx, a_gt):
    y0, x0, m = win
    ys, xs = idx // W, idx % W
    inside = (ys >= y0) & (ys < y0 + m.shape[0]) & (xs >= x0) & (xs < x0 + m.shape[1])
    if not inside.any():
        return 0.0
    i = float(m[ys[inside] - y0, xs[inside] - x0].sum())
    return i / (a_gt + float(m.sum()) - i) if i else 0.0

def seed_targets(name, seeds):
    """targets[k] = [(record, instance, iou)] per annotator set with a match; best_iou[k] = best IoU over all sets."""
    recs = GT_INDEX[name]["records"]
    tg = []
    for y0, x0, m, _, _ in seeds:
        row = []
        for r, rec in enumerate(recs):
            ious = [gt_window_iou((y0, x0, m), idx, rec["area"][j]) for j, idx in enumerate(rec["idx"])]
            if ious and max(ious) >= CFG.match_min_iou:
                j = int(np.argmax(ious))
                row.append((r, j, float(ious[j])))
        tg.append(row)
    return tg

def gt_groups(name):
    """For each (record, instance): the matching instance in every annotator set (itself included)."""
    recs = GT_INDEX[name]["records"]
    labels = []
    for rec in recs:
        v = np.zeros(H * W, np.int32)
        for j, idx in enumerate(rec["idx"]):
            v[idx] = j + 1
        labels.append(v)
    groups = {}
    for r, rec in enumerate(recs):
        for j, idx in enumerate(rec["idx"]):
            g = [(r, j, 1.0)]
            for r2, other in enumerate(recs):
                if r2 == r or not len(other["idx"]):
                    continue
                cnt = np.bincount(labels[r2][idx], minlength=len(other["idx"]) + 1)[1:].astype(np.float64)
                iou = cnt / (rec["area"][j] + other["area"] - cnt)
                if iou.max() >= CFG.match_min_iou:
                    g.append((r2, int(iou.argmax()), float(iou.max())))
            groups[(r, j)] = g
    return groups

def pool_task(task):
    name, split = task
    seeds = seed_pool(name, split, CFG.seed_det_floor)
    if split == "test":
        return name, seeds, None, None
    return name, seeds, seed_targets(name, seeds), gt_groups(name)

def pool_recall(names, src=None):
    hit3 = hit5 = total = 0
    for n in names:
        recs = GT_INDEX[n]["records"]
        best = {}
        for k, row in enumerate(TARGETS[n]):
            if src is not None and SEEDS[n][k][4] != src:
                continue
            for r, j, iou in row:
                best[(r, j)] = max(best.get((r, j), 0.0), iou)
        total += sum(len(rec["idx"]) for rec in recs)
        hit3 += sum(v > 0.3 for v in best.values()); hit5 += sum(v > 0.5 for v in best.values())
    return {"recall@0.3": hit3 / max(total, 1), "recall@0.5": hit5 / max(total, 1), "filaments": total}

SEEDS, TARGETS, GROUPS = {}, {}, {}
_jobs = [(n, "train") for n in ALL_NAMES] if CFG.stage == "train" else [(n, "test") for n in TEST_NAMES] if CFG.stage == "test" else []
if _jobs:
    _pool = ProcessPoolExecutor(max_workers=CPU_POOL, mp_context=mp.get_context("fork"))
    for name, seeds, tg, groups in tqdm(_pool.map(pool_task, _jobs, chunksize=2), total=len(_jobs), desc=f"Seed pools ({CFG.stage})"):
        SEEDS[name] = seeds
        if tg is not None:
            TARGETS[name], GROUPS[name] = tg, groups
    _pool.shutdown()
print("Seed geometry:", SEED_GEOM, "| prefilter:", SEED_PRE, "| seeds per image:",
      round(float(np.mean([len(v) for v in SEEDS.values()])), 1) if SEEDS else None)
if CFG.stage == "train":
    SEED_STATS = {"all": pool_recall(ALL_NAMES), "v3": pool_recall(ALL_NAMES, 0), "det_added": pool_recall(ALL_NAMES, 1)}
    print("Seed recall:", SEED_STATS)
    write_json(RUN / "seed_stats.json", SEED_STATS)


## Cell 11 — Crop dataset (seed -> 5-channel crop, soft consensus mask, spine heatmap), D4 augmentation (pilot code)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

S = CFG.crop_size
FG_CACHE = {}

def maps_of(name, split="train"):
    img, flat = cached_arrays(cache_key(split, name), False)
    fg = np.load((OOF_PATHS if split == "train" else TEST_MAPS)[name][0], mmap_mode="r")
    return img, flat, fg

def crop_window(y0, x0, h, w, rng=None):
    cy, cx = y0 + h / 2.0, x0 + w / 2.0
    s = max(CFG.min_crop, CFG.context * max(h, w))
    if rng is not None:
        s *= rng.uniform(0.9, 1.2)
        cy += rng.uniform(-0.08, 0.08) * s
        cx += rng.uniform(-0.08, 0.08) * s
    s = int(round(min(max(s, CFG.min_crop), CFG.max_crop)))
    return int(round(cy - s / 2.0)), int(round(cx - s / 2.0)), s

def extract(arr, ty, tx, s, fill):
    out = np.full((s, s), fill, np.asarray(arr[:1, :1]).dtype)
    ya, xa, yb, xb = max(ty, 0), max(tx, 0), min(ty + s, H), min(tx + s, W)
    if ya < yb and xa < xb:
        out[ya - ty:yb - ty, xa - tx:xb - tx] = arr[ya:yb, xa:xb]
    return out

def paste(win, ty, tx, s):
    y0, x0, m = win
    out = np.zeros((s, s), np.float32)
    ya, xa = max(y0, ty), max(x0, tx)
    yb, xb = min(y0 + m.shape[0], ty + s), min(x0 + m.shape[1], tx + s)
    if ya < yb and xa < xb:
        out[ya - ty:yb - ty, xa - tx:xb - tx] = m[ya - y0:yb - y0, xa - x0:xb - x0]
    return out

def gt_local(idx, ty, tx, s):
    out = np.zeros((s, s), np.float32)
    ys, xs = idx // W - ty, idx % W - tx
    ok = (ys >= 0) & (ys < s) & (xs >= 0) & (xs < s)
    out[ys[ok], xs[ok]] = 1.0
    return out

def spine_local(pts, ty, tx, s):
    out = np.zeros((s, s), np.uint8)
    if len(pts) >= 2:
        q = np.rint(pts - np.array([tx, ty])).astype(np.int32).reshape(-1, 1, 2)
        cv2.polylines(out, [q], False, 1, thickness=CFG.spine_width)
    return out.astype(np.float32)

def resize(a, interp=None):
    if a.shape[0] == S:
        return a.astype(np.float32)
    if interp is None:
        interp = cv2.INTER_AREA if a.shape[0] > S else cv2.INTER_LINEAR
    return cv2.resize(a.astype(np.float32), (S, S), interpolation=interp)

def make_input(name, seed_win, ty, tx, s, split="train"):
    img, flat, fg = maps_of(name, split)
    x = np.stack([
        (resize(extract(img, ty, tx, s, 0)) / 255.0 - 0.5) / 0.25,
        (resize(extract(flat, ty, tx, s, 128)) / 128.0 - 1.0) / 0.15,
        (np.clip(resize(extract(RADIAL, ty, tx, s, 1.3)), 0, 1.25) - 0.6) / 0.35,
        resize(paste(seed_win, ty, tx, s), cv2.INTER_LINEAR),
        resize(extract(fg, ty, tx, s, 0)) / 255.0])
    return x.astype(np.float32)

def make_target(name, members, ty, tx, s):
    """Mean over every annotator set of the member instance it has (zeros for sets without one)."""
    recs = GT_INDEX[name]["records"]
    m = np.zeros((s, s), np.float32); sp = np.zeros((s, s), np.float32)
    for r, j, _ in members:
        m += gt_local(recs[r]["idx"][j], ty, tx, s)
        sp = np.maximum(sp, spine_local(SPINES[name][r][j], ty, tx, s))
    m /= max(len(recs), 1)
    return np.stack([resize(m), resize(sp, cv2.INTER_LINEAR)]).astype(np.float32)

def jitter_gt_seed(name, r, j, rng):
    idx = GT_INDEX[name]["records"][r]["idx"][j]
    ys, xs = idx // W, idx % W
    y0, x0 = ys.min(), xs.min()
    m = np.zeros((ys.max() - y0 + 1, xs.max() - x0 + 1), np.uint8)
    m[ys - y0, xs - x0] = 1
    k = int(rng.integers(-CFG.jitter_morph_px, CFG.jitter_morph_px + 1))
    if k:
        pad = abs(k) + 1
        m = np.pad(m, pad); y0 -= pad; x0 -= pad
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * abs(k) + 1, 2 * abs(k) + 1))
        m = cv2.dilate(m, ker) if k > 0 else cv2.erode(m, ker)
    if rng.random() < 0.2 and m.shape[1] > 8:                 # partial seed: keep one end
        cut = int(m.shape[1] * rng.uniform(0.4, 0.8))
        m[:, cut:] = 0 if rng.random() < 0.5 else m[:, cut:]
    if not m.any():
        m[m.shape[0] // 2, m.shape[1] // 2] = 1
    dy, dx = rng.integers(-CFG.jitter_seed_px, CFG.jitter_seed_px + 1, size=2)
    return int(y0 + dy), int(x0 + dx), m.astype(bool)

def d4(a, op):
    k, flip = op % 4, op >= 4
    a = np.rot90(a, k, axes=(-2, -1))
    return np.ascontiguousarray(a[..., ::-1] if flip else a)

def train_lists(names):
    return ([(n, k) for n in names for k in range(len(SEEDS[n]))], [(n, r, j) for n in names for (r, j) in GROUPS[n]])

TRAIN_REAL, TRAIN_GT = train_lists(ALL_NAMES if CFG.stage == "train" else [])

class CropTrain(Dataset):
    def __init__(self, n, epoch):
        self.n, self.epoch = n, epoch
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        rng = np.random.default_rng((CFG.seed, self.epoch, i))
        if rng.random() < CFG.gt_seed_fraction:
            name, r, j = TRAIN_GT[int(rng.integers(len(TRAIN_GT)))]
            win = jitter_gt_seed(name, r, j, rng)
            members = GROUPS[name][(r, j)]
        else:
            name, k = TRAIN_REAL[int(rng.integers(len(TRAIN_REAL)))]
            y0, x0, m, _, _ = SEEDS[name][k]
            win, members = (y0, x0, m), TARGETS[name][k]
        ty, tx, s = crop_window(win[0], win[1], win[2].shape[0], win[2].shape[1], rng)
        x, y = make_input(name, win, ty, tx, s), make_target(name, members, ty, tx, s)
        op = int(rng.integers(8))
        x, y = d4(x, op), d4(y, op)
        if rng.random() < 0.5:                                     # intensity jitter on the two image channels
            x[:2] = x[:2] * rng.uniform(0.85, 1.15) + rng.uniform(-0.15, 0.15)
        return torch.from_numpy(x), torch.from_numpy(y)

def _alignment_test(n=64):
    """Geometry of crop, paste and resize: the exact ground-truth mask used as seed must coincide with its target."""
    rng = np.random.default_rng(1)
    exact, jit = [], []
    for t in range(n):
        name, r, j = TRAIN_GT[int(rng.integers(len(TRAIN_GT)))]
        idx = GT_INDEX[name]["records"][r]["idx"][j]
        ys, xs = idx // W, idx % W
        m = np.zeros((ys.max() - ys.min() + 1, xs.max() - xs.min() + 1), bool)
        m[ys - ys.min(), xs - xs.min()] = True
        for win, store in (((int(ys.min()), int(xs.min()), m), exact), (jitter_gt_seed(name, r, j, rng), jit)):
            ty, tx, s = crop_window(win[0], win[1], win[2].shape[0], win[2].shape[1])
            x, y = make_input(name, win, ty, tx, s), make_target(name, [(r, j, 1.0)], ty, tx, s)
            a, b = x[3] > 0.5, y[0] > 0.5 * max(float(y[0].max()), 1e-6)
            store.append((a & b).sum() / max((a | b).sum(), 1))
    print(f"Alignment test over {n} crops: IoU(exact seed, target) {np.mean(exact):.3f} | IoU(jittered seed, target) {np.mean(jit):.3f}")
    assert np.mean(exact) > 0.9, "crop / target geometry is misaligned"

if CFG.stage == "train":
    _alignment_test()
    _probe = CropTrain(4, 0)[0]
    print("All images: real seeds", len(TRAIN_REAL), "| ground-truth instances", len(TRAIN_GT),
          "| crop tensors", tuple(_probe[0].shape), tuple(_probe[1].shape))


## Cell 12 — Refiner model and training per fold (pilot recipe; resumable through `done.json` + `best.pt`)


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda" and not CFG.debug_run and CFG.stage != "score":
    raise RuntimeError("Training the refiner needs a GPU runtime.")
AMP = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

def build_model(pretrained=True):
    return smp.Unet(encoder_name=CFG.encoder, encoder_weights="imagenet" if pretrained else None,
                    in_channels=5, classes=2, decoder_attention_type="scse")

def soft_dice(logits, target):
    p = torch.sigmoid(logits)
    inter = (p * target).sum((1, 2, 3))
    return (1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)).mean()

def loss_fn(out, y):
    m, sp = out[:, :1].float(), out[:, 1:].float()
    return (F.binary_cross_entropy_with_logits(m, y[:, :1]) + soft_dice(m, y[:, :1])
            + CFG.spine_weight * F.binary_cross_entropy_with_logits(sp, y[:, 1:], pos_weight=torch.tensor(5.0, device=sp.device)))

def train_refiner(fold):
    global TRAIN_REAL, TRAIN_GT
    FOLD_DIR = RUN / f"fold{fold}"
    FOLD_DIR.mkdir(parents=True, exist_ok=True)
    done = FOLD_DIR / "done.json"
    if done.is_file() and (FOLD_DIR / "best.pt").is_file():
        print("Refiner already trained:", json.loads(done.read_text()))
        return
    TRAIN_REAL, TRAIN_GT = train_lists([n for n in ALL_NAMES if FOLD_OF[n] != fold])   # workers fork after this
    print(f"Fold {fold}: training on {len(TRAIN_REAL)} real seeds and {len(TRAIN_GT)} ground-truth instances")
    torch.manual_seed(CFG.seed + fold)
    model = build_model().to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    steps = CFG.epochs * math.ceil(CFG.samples_per_epoch / CFG.batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.lr, total_steps=steps, pct_start=0.1)
    hist, t0 = [], time.time()
    for ep in range(CFG.epochs):
        model.train()
        dl = DataLoader(CropTrain(CFG.samples_per_epoch, ep), batch_size=CFG.batch, shuffle=False,
                        num_workers=CFG.workers, pin_memory=True, drop_last=True, persistent_workers=False)
        tot, nb, tw = 0.0, 0, time.time()
        for x, y in dl:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE, dtype=AMP, enabled=DEVICE == "cuda"):
                out = model(x)
            loss = loss_fn(out, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            tot += float(loss); nb += 1
        hist.append({"epoch": ep + 1, "loss": tot / max(nb, 1), "seconds": time.time() - tw})
        print(f"fold {fold} epoch {ep + 1}/{CFG.epochs} loss {tot / max(nb, 1):.4f} ({time.time() - tw:.0f}s)")
    torch.save({"state": model.state_dict(), "config": asdict(CFG)}, FOLD_DIR / "best.pt")
    pd.DataFrame(hist).to_csv(FOLD_DIR / "history.csv", index=False)
    write_json(done, {"fold": fold, "minutes": (time.time() - t0) / 60, "epochs": CFG.epochs,
                      "sha256": hash_file(FOLD_DIR / "best.pt")})
    push(f"fold{fold}")
    del model, opt
    gc.collect(); torch.cuda.empty_cache()


## Cell 13 — Refinement: each training image by the refiner of its own fold (out-of-fold), saved per fold
Features per refined candidate: V3's 12 mask features, V4 and V5 detector evidence (score, IoU, coverage), refiner mean and peak
probability, seed score and source, IoU between the refined mask and its seed.


In [ ]:
TTA_OPS = [int(o[1]) + (4 if o.endswith("f") else 0) for o in CFG.tta]
REF_INFO = ("ref_mean_p", "ref_max_p", "seed_score", "seed_src", "seed_iou")
REF_FEATURES = tuple(FEATURES) + ("v4_score", "v4_iou", "v4_cov", "v5_score", "v5_iou", "v5_cov") + REF_INFO

def load_refiner(fold):
    m = build_model(pretrained=False)
    m.load_state_dict(torch.load(RUN / f"fold{fold}" / "best.pt", map_location="cpu")["state"])
    return m.to(DEVICE).to(memory_format=torch.channels_last).eval()

@torch.no_grad()
def refine_probs(xs, models):
    """xs [n, 5, S, S] -> mask probability [n, S, S], mean over models and D4 ops."""
    out = np.zeros((len(xs), S, S), np.float32)
    for op in TTA_OPS:
        xb = torch.from_numpy(np.stack([d4(x, op) for x in xs])).to(DEVICE).to(memory_format=torch.channels_last)
        for model in models:
            with torch.autocast(DEVICE, dtype=AMP, enabled=DEVICE == "cuda"):
                y = torch.sigmoid(model(xb).float())[:, 0].cpu().numpy()
            y = y[..., ::-1] if op >= 4 else y
            out += np.rot90(y, -(op % 4), axes=(-2, -1))
    return out / (len(TTA_OPS) * len(models))

def refine_image(name, models, split):
    seeds = SEEDS[name]
    if not seeds:
        return np.zeros((H, W), np.int32), 0, np.zeros((0, len(REF_INFO)), np.float32)
    wins = [crop_window(y0, x0, m.shape[0], m.shape[1]) for y0, x0, m, _, _ in seeds]
    xs = np.stack([make_input(name, (y0, x0, m), *w, split) for (y0, x0, m, _, _), w in zip(seeds, wins)])
    probs = np.concatenate([refine_probs(xs[i:i + 128], models) for i in range(0, len(xs), 128)])
    cands = []
    for (y0, x0, m, score, src), (ty, tx, s), p in zip(seeds, wins, probs):
        ps = cv2.resize(p, (s, s), interpolation=cv2.INTER_LINEAR)
        peak = float(ps.max())
        if peak < CFG.min_peak:
            continue
        mask = ps >= CFG.prob_threshold * peak
        seedl = paste((y0, x0, m), ty, tx, s) > 0
        lab, nl = ndi.label(mask, structure=STRUCT8)
        if nl > 1:
            ov = np.bincount(lab[seedl], minlength=nl + 1)[1:]
            keep = int(ov.argmax()) + 1 if ov.max() > 0 else int(np.bincount(lab.ravel())[1:].argmax()) + 1
            mask = lab == keep
        siou = float((mask & seedl).sum() / max((mask | seedl).sum(), 1))
        cands.append((float(ps[mask].mean()), peak, ty, tx, mask, score, src, siou))
    cands.sort(key=lambda c: -c[0])
    out = np.zeros((H, W), np.int32); taken = np.zeros((H, W), bool); info = []
    for mean_p, max_p, ty, tx, mask, score, src, siou in cands:
        ya, xa, yb, xb = max(ty, 0), max(tx, 0), min(ty + mask.shape[0], H), min(tx + mask.shape[1], W)
        mk = mask[ya - ty:yb - ty, xa - tx:xb - tx]
        free = mk & ~taken[ya:yb, xa:xb]
        if free.sum() < max(CFG.min_area, CFG.dup_keep_frac * mk.sum()):
            continue
        out[ya:yb, xa:xb][free] = len(info) + 1
        taken[ya:yb, xa:xb] |= free
        info.append((mean_p, max_p, score, src, siou))
    return out, len(info), np.asarray(info, np.float32).reshape(-1, len(REF_INFO))

def evidence(lab, n, area, det_lists):
    ev = np.zeros((n, N_DETF))
    for dets in det_lists:
        ev += np.stack(detection_evidence(lab, n, area, rasterize_detections(dets)), 1) if n else 0
    return ev / max(len(det_lists), 1)

def det_split(name, split):
    return ([DET_V4[name]], [DET_V5[name]]) if split == "train" else (DET_V4_TEST[name], DET_V5_TEST[name])

def rles_of(lab, n):
    return [mask_api.encode(np.asfortranarray((lab == i).astype(np.uint8)))["counts"].decode("ascii") for i in range(1, n + 1)]

def refined_record(name, models, split):
    lab, n, info = refine_image(name, models, split)
    fg, ct, img, flat = maps_for(name, split)
    feats, area = component_features(lab, n, fg, ct, img, flat)
    d4l, d5l = det_split(name, split)
    X = np.concatenate([feats, evidence(lab, n, area, d4l), evidence(lab, n, area, d5l), info], 1).astype(np.float32)
    return {"rles": rles_of(lab, n), "X": X, "info": info}

def _identity_check(names):
    ious = []
    for name in names:
        for y0, x0, m, _, _ in SEEDS[name]:
            ty, tx, s = crop_window(y0, x0, m.shape[0], m.shape[1])
            ps = cv2.resize(make_input(name, (y0, x0, m), ty, tx, s)[3], (s, s), interpolation=cv2.INTER_LINEAR)
            back = ps >= CFG.prob_threshold * max(float(ps.max()), 1e-6)
            ref = paste((y0, x0, m), ty, tx, s) > 0
            ious.append((back & ref).sum() / max((back | ref).sum(), 1))
    print(f"Identity check: IoU(seed after the round trip, seed) {np.mean(ious):.3f} over {len(ious)} seeds")
    assert np.mean(ious) > 0.85

if CFG.stage == "train":
    _identity_check(ALL_NAMES[:6])
    for fold in CFG.folds:
        train_refiner(fold)
        out_path = RUN / f"refined_oof_fold{fold}.pkl"
        if out_path.is_file():
            print(f"Fold {fold}: refined candidates already saved")
            continue
        models = [load_refiner(fold)]
        names = [n for n in ALL_NAMES if FOLD_OF[n] == fold]
        rec = {n: refined_record(n, models, "train") for n in tqdm(names, desc=f"Refine fold {fold} (out-of-fold)")}
        save_pickle(out_path, rec)
        push()
        print(f"Fold {fold}: {np.mean([len(r['rles']) for r in rec.values()]):.2f} refined candidates per image")
        del models
        gc.collect(); torch.cuda.empty_cache()


## Cell 14 — Test refinement: every test image by all 4 refiners (probabilities averaged)


In [ ]:
if CFG.stage == "test":
    _folds = sorted(int(p.parent.name.replace("fold", "")) for p in RUN.glob("fold*/done.json"))
    if not CFG.debug_run and _folds != list(range(CFG.n_folds)):
        raise RuntimeError(f"All {CFG.n_folds} refiners are needed for the test stage; found {_folds}")
    models = [load_refiner(f) for f in _folds]
    rec = {n: refined_record(n, models, "test") for n in tqdm(TEST_NAMES, desc=f"Refine test ({len(models)} refiners)")}
    save_pickle(RUN / "refined_test.pkl", rec)
    push()
    print(f"Test: {np.mean([len(r['rles']) for r in rec.values()]):.2f} refined candidates per image")


## Cell 15 — Scoring (CPU): V4 replica, refined only, union; cross-fitted over 4 folds; adoption rule; test exports


In [ ]:
def decode_labels(rles):
    lab = np.zeros((H, W), np.int32)
    for i, r in enumerate(rles, 1):
        lab[decode_mask({"size": [H, W], "counts": r})] = i
    return lab

def targets_and_ious(name, lab, n, area):
    mats = overlap_matrices(lab, n, area, GT_INDEX[name])
    y = np.zeros(n)
    for _, iou in mats:
        if iou.shape[0] and n:
            b = iou.max(0)
            y += np.where(b > 0.5, b, 0.0)
    return y / max(len(mats), 1), [iou for _, iou in mats], len(mats)

def variant_rows(task):
    """One image -> {variant: (X, rles, y, ious, k)} for v4, v5 and union (train split also gets targets)."""
    name, split, ref = task
    fg, ct, img, flat = maps_for(name, split)
    d4l, d5l = det_split(name, split)
    out = {}
    lab4, n4, n4v, X4, area4 = build_candidates(fg, ct, img, flat, d4l)               # V4 v4_full candidates
    out["v4"] = (lab4, n4, X4, area4)
    lab5 = decode_labels(ref["rles"]); n5 = len(ref["rles"])
    feats5, area5 = component_features(lab5, n5, fg, ct, img, flat)
    X5 = np.concatenate([ref["X"], np.zeros((n5, 1), np.float32)], 1)                 # src 0 = refined
    out["v5"] = (lab5, n5, X5, area5)
    lab_u, taken, extra_src = lab5.copy(), lab5 > 0, []
    for k in range(1, n4 + 1):                                                         # V4 candidates on free pixels
        m = lab4 == k
        free = m & ~taken
        if free.sum() >= max(CFG.min_area, CFG.union_keep_frac * m.sum()):
            lab_u[free] = n5 + len(extra_src) + 1
            taken |= free
            extra_src.append(1.0 + X4[k - 1, -1])                                      # 1 = V4 UNet, 2/3 = V4 detector-only
    nu = n5 + len(extra_src)
    featsu, areau = component_features(lab_u, nu, fg, ct, img, flat)
    info_u = np.full((nu, len(REF_INFO)), np.nan, np.float32); info_u[:n5] = ref["info"]
    Xu = np.concatenate([featsu, evidence(lab_u, nu, areau, d4l), evidence(lab_u, nu, areau, d5l), info_u,
                         np.r_[np.zeros(n5), extra_src][:, None]], 1).astype(np.float32)
    out["union"] = (lab_u, nu, Xu, areau)
    res = {}
    for v, (lab, n, X, area) in out.items():
        if split == "train":
            y, ious, k = targets_and_ious(name, lab, n, area)
            res[v] = (X, None, y, ious, k)
        else:
            res[v] = (X, rles_of(lab, n), None, None, None)
    return name, res

def make_hgb():
    return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
                                         min_samples_leaf=40, l2_regularization=1.0, random_state=CFG.seed)

def image_stats(u, off, ious, tau):
    out = np.zeros((len(ious), 5))
    for i in range(len(ious)):
        keep = u[off[i]:off[i + 1]] >= tau
        for iou in ious[i]:
            out[i] += match_stats(iou[:, keep])
    return out

def pq_value(st):
    den = st[1] + 0.5 * (st[2] + st[3])
    return float(st[0] / den) if den else 0.0

def paired_bootstrap(a, b, n=None):
    n = CFG.bootstrap if n is None else n
    rng = np.random.default_rng(CFG.seed)
    w = rng.multinomial(len(a), np.full(len(a), 1.0 / len(a)), size=n).astype(np.float64)
    sa, sb = w @ a[:, :4], w @ b[:, :4]
    pq = lambda s: s[:, 0] / np.maximum(s[:, 1] + 0.5 * (s[:, 2] + s[:, 3]), 1e-9)
    return np.percentile(pq(sa) - pq(sb), [2.5, 97.5])

if CFG.stage == "score":
    REF_OOF = {}
    for f in range(CFG.n_folds):
        p = RUN / f"refined_oof_fold{f}.pkl"
        if p.is_file():
            REF_OOF.update(load_pickle(p))
    REF_TEST = load_pickle(RUN / "refined_test.pkl") if (RUN / "refined_test.pkl").is_file() else {}
    names = [n for n in ALL_NAMES if n in REF_OOF]
    if not CFG.debug_run and len(names) != len(ALL_NAMES):
        raise RuntimeError(f"Refined out-of-fold candidates cover {len(names)} of {len(ALL_NAMES)} images")
    img_fold = np.array([FOLD_OF[n] for n in names])
    pool = ProcessPoolExecutor(max_workers=CPU_POOL, mp_context=mp.get_context("fork"))
    rows = dict(tqdm(pool.map(variant_rows, [(n, "train", REF_OOF[n]) for n in names], chunksize=2),
                     total=len(names), desc="Variant candidates (out-of-fold)"))
    TAUS = np.array(CFG.scorer_taus)
    RES = {}
    for v in ("v4", "v5", "union"):
        X = np.concatenate([rows[n][v][0] for n in names]); Y = np.concatenate([rows[n][v][2] for n in names])
        cnt = [len(rows[n][v][2]) for n in names]
        off = np.r_[0, np.cumsum(cnt)]
        K = np.repeat([rows[n][v][4] for n in names], cnt).astype(np.float64)
        g = np.repeat(img_fold, cnt)
        ious = [rows[n][v][3] for n in names]
        u = np.zeros(len(Y))
        for f in np.unique(img_fold):
            tr, te = g != f, g == f
            u[te] = make_hgb().fit(X[tr], Y[tr], sample_weight=K[tr]).predict(X[te])
        stats_all = [image_stats(u, off, ious, t) for t in TAUS]
        pqs = np.array([pq_value(s.sum(0)) for s in stats_all])
        b = int(np.lexsort((TAUS, -pqs))[0])
        RES[v] = {"tau": float(TAUS[b]), "stats": stats_all[b], "model": make_hgb().fit(X, Y, sample_weight=K),
                  "candidates": len(Y), "u": u, "off": off}
        s = summarize(stats_all[b].sum(0))
        print(f"{v:6s}: PQ {s['pq']:.4f} SQ {s['sq']:.3f} RQ {s['rq']:.3f} P {s['precision']:.3f} R {s['recall']:.3f}"
              f" tau {TAUS[b]:g} | {len(Y)} candidates")
    base = RES["v4"]["stats"]
    table = []
    for v, r in RES.items():
        s = summarize(r["stats"].sum(0))
        wins = sum(pq_value(r["stats"][img_fold == f].sum(0)) > pq_value(base[img_fold == f].sum(0)) + 1e-12
                   for f in np.unique(img_fold))
        lo, hi = paired_bootstrap(r["stats"], base) if v != "v4" else (0.0, 0.0)
        table.append({"variant": v, "tau": r["tau"], **{k: s[k] for k in ("pq", "sq", "rq", "precision", "recall", "tp", "fp", "fn")},
                      "candidates": r["candidates"], "gain": s["pq"] - pq_value(base.sum(0)), "ci_low": float(lo),
                      "ci_high": float(hi), "fold_wins": int(wins)})
    TABLE = pd.DataFrame(table)
    TABLE.to_csv(RUN / "variants.csv", index=False)
    print("\n" + TABLE.round(4).to_string(index=False))
    ok = TABLE[(TABLE.variant != "v4") & (TABLE.gain >= CFG.min_gain) & (TABLE.fold_wins >= min(CFG.min_fold_wins, CFG.n_folds))]
    FINAL = ok.sort_values("pq", ascending=False).variant.iloc[0] if len(ok) else "v4"
    print("\nAdopted:", FINAL)

    EXPORTS = {}
    if REF_TEST:
        trow = dict(tqdm(pool.map(variant_rows, [(n, "test", REF_TEST[n]) for n in TEST_NAMES], chunksize=1),
                         total=len(TEST_NAMES), desc="Variant candidates (test)"))
        (RUN / "test_export").mkdir(parents=True, exist_ok=True)
        for v, r in RES.items():
            out_rows, filled = [], 0
            for n in TEST_NAMES:
                X, rles = trow[n][v][0], trow[n][v][1]
                u = r["model"].predict(X) if len(X) else np.zeros(0)
                keep = np.flatnonzero(u >= r["tau"])
                if CFG.never_empty and len(keep) == 0 and len(u):
                    keep, filled = np.array([int(np.argmax(u))]), filled + 1
                out_rows += [(f"{Path(n).stem}_{j}", rles[i]) for j, i in enumerate(keep, 1)]
            path = RUN / "test_export" / f"submission_{v}.csv"
            pd.DataFrame(out_rows, columns=["filament_id", "segmentation_rle"]).to_csv(path, index=False)
            seen_ok = True
            for stem, grp in pd.DataFrame(out_rows, columns=["filament_id", "segmentation_rle"]).groupby(
                    lambda i: out_rows[i][0].rsplit("_", 1)[0]):
                seen = np.zeros((H, W), bool)
                for rle in grp.segmentation_rle:
                    m = decode_mask({"size": [H, W], "counts": rle})
                    if not m.any() or (seen & m).any():
                        seen_ok = False
                    seen |= m
            assert seen_ok, f"{v}: empty or overlapping masks"
            EXPORTS[v] = {"path": str(path), "rows": len(out_rows), "filled": filled, "tau": r["tau"]}
            print(f"{v:6s}: {len(out_rows)} filaments ({len(out_rows) / len(TEST_NAMES):.2f} per image), {filled} filled by never-empty")
        shutil.copyfile(RUN / "test_export" / f"submission_{FINAL}.csv", RUN / "test_export" / "submission.csv")
    pool.shutdown()
    write_json(RUN / "manifest_cv.json", {"variants": TABLE.to_dict("records"), "adopted": FINAL, "exports": EXPORTS,
                                          "config": asdict(CFG), "minutes": (time.time() - T0) / 60})
    push()
    print("Artifacts:", RUN, "| minutes:", round((time.time() - T0) / 60, 1))
